# 05_silver_delta_features

## Purpose
Demonstrate Delta Lake features on Silver tables:
- **ACID Transactions**
- **Time Travel** - Query historical versions
- **DESCRIBE HISTORY** - View table history
- **RESTORE** - Rollback to previous version
- **VACUUM** - Clean up old files

## Day 5 Deliverable
This notebook provides evidence of Delta Lake capabilities for documentation.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CAT = "zillow"
SILVER = "zillow_silver"

# Test table - using zip_ts_silver as example
TEST_TABLE = f"{CAT}.{SILVER}.zip_ts_silver"

In [0]:
# ACID TRANSACTIONS DEMONSTRATION
print("="*60)
print("1. ACID TRANSACTIONS")
print("="*60)

print("""
Delta Lake ensures ACID properties:
- Atomicity: All or nothing - transactions complete fully or not at all
- Consistency: Data integrity constraints maintained
- Isolation: Concurrent transactions don't interfere
- Durability: Committed transactions are permanent

Example: Multiple writers can safely update the same table.
""")

# Show current table info
try:
    df = spark.table(TEST_TABLE)
    count = df.count()
    print(f"Current row count in {TEST_TABLE}: {count:,}")
    
    # Show table properties
    display(spark.sql(f"DESCRIBE EXTENDED {TEST_TABLE}"))
except Exception as e:
    print(f"Table not found: {e}")
    print("Run Silver transformations first to create the table.")

In [0]:
# TABLE HISTORY
print("\n" + "="*60)
print("2. TABLE HISTORY (DESCRIBE HISTORY)")
print("="*60)

print("""
DESCRIBE HISTORY shows all operations performed on a Delta table:
- Version number
- Timestamp
- Operation (WRITE, UPDATE, DELETE, MERGE, etc.)
- Who performed it
- Metrics (rows affected)
""")

try:
    history_df = spark.sql(f"DESCRIBE HISTORY {TEST_TABLE}")
    print(f"\nHistory for {TEST_TABLE}:")
    display(history_df.select(
        "version", 
        "timestamp", 
        "operation", 
        "operationParameters",
        "operationMetrics"
    ))
except Exception as e:
    print(f"Error: {e}")

In [0]:
# TIME TRAVEL
print("\n" + "="*60)
print("3. TIME TRAVEL")
print("="*60)

print("""
Time travel allows querying historical versions of data:
- Query by VERSION: SELECT * FROM table VERSION AS OF 1
- Query by TIMESTAMP: SELECT * FROM table TIMESTAMP AS OF '2024-01-01'

This is useful for:
- Auditing changes
- Debugging
- Reproducing ML experiments
- Comparing data states
""")

try:
    # Get available versions
    history = spark.sql(f"DESCRIBE HISTORY {TEST_TABLE}")
    versions = [row.version for row in history.select("version").collect()]
    
    if len(versions) > 0:
        current_version = versions[0]
        print(f"\nCurrent version: {current_version}")
        
        # Query current version
        current_count = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM {TEST_TABLE} VERSION AS OF {current_version}
        """).collect()[0]["cnt"]
        print(f"Rows in version {current_version}: {current_count:,}")
        
        # If multiple versions exist, show older version
        if len(versions) > 1:
            old_version = versions[-1]
            old_count = spark.sql(f"""
                SELECT COUNT(*) as cnt FROM {TEST_TABLE} VERSION AS OF {old_version}
            """).collect()[0]["cnt"]
            print(f"Rows in version {old_version}: {old_count:,}")
            
            print(f"\nChange: {current_count - old_count:+,} rows")
        else:
            print("\nOnly one version exists. Perform more operations to see time travel in action.")
            
except Exception as e:
    print(f"Error: {e}")

In [0]:
# TIME TRAVEL: COMPARE VERSIONS
print("\n" + "="*60)
print("4. COMPARING DATA BETWEEN VERSIONS")
print("="*60)

print("""
You can compare data between versions to see what changed.
This is useful for auditing and debugging.
""")

try:
    history = spark.sql(f"DESCRIBE HISTORY {TEST_TABLE}")
    versions = [row.version for row in history.select("version").collect()]
    
    if len(versions) >= 2:
        v_new = versions[0]
        v_old = versions[-1]
        
        # Find new records (in new version but not in old)
        new_records = spark.sql(f"""
            SELECT * FROM {TEST_TABLE} VERSION AS OF {v_new}
            EXCEPT
            SELECT * FROM {TEST_TABLE} VERSION AS OF {v_old}
        """)
        
        print(f"\nRecords added between version {v_old} and {v_new}: {new_records.count():,}")
        
        # Show sample
        if new_records.count() > 0:
            print("\nSample of changed records:")
            display(new_records.limit(5))
    else:
        print("Need at least 2 versions to compare.")
        
except Exception as e:
    print(f"Error: {e}")

In [0]:
# RESTORE DEMONSTRATION (explanation only, not executing)
print("\n" + "="*60)
print("5. RESTORE TABLE (Rollback)")
print("="*60)

print("""
RESTORE TABLE allows rolling back to a previous version:

  -- Restore to specific version
  RESTORE TABLE my_table TO VERSION AS OF 5
  
  -- Restore to specific timestamp
  RESTORE TABLE my_table TO TIMESTAMP AS OF '2024-01-01 00:00:00'

This is useful for:
- Recovering from accidental deletes
- Undoing bad updates
- Testing with previous data states

Note: RESTORE creates a new version, so you can still access
all previous versions including the "bad" one.
""")

print("\n[RESTORE command not executed to preserve current data]")
print("\nTo restore, run:")
print(f"  RESTORE TABLE {TEST_TABLE} TO VERSION AS OF <version_number>")

In [0]:
# VACUUM DEMONSTRATION
print("\n" + "="*60)
print("6. VACUUM (Cleanup Old Files)")
print("="*60)

print("""
VACUUM removes old data files that are no longer needed:

  -- Remove files older than 7 days (default)
  VACUUM my_table
  
  -- Remove files older than 24 hours
  VACUUM my_table RETAIN 24 HOURS
  
  -- Dry run (show what would be deleted)
  VACUUM my_table DRY RUN

Important:
- Default retention is 7 days
- Cannot use time travel for data older than VACUUM threshold
- Run VACUUM regularly to manage storage costs
""")

# Show dry run
try:
    print(f"\nDry run for {TEST_TABLE}:")
    display(spark.sql(f"VACUUM {TEST_TABLE} DRY RUN"))
except Exception as e:
    print(f"Error: {e}")

In [0]:
# SUMMARY
print("\n" + "="*60)
print("DELTA LAKE FEATURES SUMMARY")
print("="*60)

print("""
╔══════════════════════════════════════════════════════════╗
║                    DELTA LAKE FEATURES                    ║
╠══════════════════════════════════════════════════════════╣
║  Feature          │ Command                              ║
╠═══════════════════╪══════════════════════════════════════╣
║  View History     │ DESCRIBE HISTORY table_name          ║
║  Time Travel      │ SELECT * FROM table VERSION AS OF n  ║
║  Timestamp Query  │ SELECT * FROM table TIMESTAMP AS OF  ║
║  Rollback         │ RESTORE TABLE table TO VERSION AS OF ║
║  Cleanup          │ VACUUM table RETAIN n HOURS          ║
║  Optimize         │ OPTIMIZE table ZORDER BY (cols)      ║
╚══════════════════════════════════════════════════════════╝

All Silver tables in this project are Delta tables with:
✅ Full version history
✅ Time travel capability
✅ ACID transaction support
✅ Schema enforcement
""")